In [19]:
import pandas as pd

#wrangler_sample_df = pd.read_csv("https://aka.ms/wrangler/titanic.csv")
#display(wrangler_sample_df)
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 21, Finished, Available, Finished, False)

In [20]:
#defining variables
landing_path = "Files/Landing/netflix_titles.csv"

bronze_table = "bronze_netflix_titles"

batch_id = "20260716_001"
print(landing_path)

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 22, Finished, Available, Finished, False)

Files/Landing/netflix_titles.csv


In [21]:
#Reading CSV file
print(landing_path)
df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(landing_path)
)



StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 23, Finished, Available, Finished, False)

Files/Landing/netflix_titles.csv


In [22]:
#display(df)
df = df.filter(~df.show_id.contains('"'))
df = df.where(df.show_id != "s8420")
df.createOrReplaceTempView("bronze_layer")
#spark.sql("SELECT * FROM bronze_layer order by show_id limit 10").show()

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 24, Finished, Available, Finished, False)

In [23]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 25, Finished, Available, Finished, False)

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|      0|   0|    0|    2634| 824|    830|        11|           0|     4|       3|        0|          0|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [24]:
df_bronze = (
    df.withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "batch_id",
        lit(batch_id)
    )
    .withColumn(
        "source_file",
        input_file_name()
    )
)

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 26, Finished, Available, Finished, False)

In [25]:
#display(df_bronze.limit(10))
invalid_rows = df_bronze.filter(
    ~col("show_id").startswith("s")
)
display(invalid_rows)

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2407fea1-f6c3-461e-a26c-927fd19c8fe0)

In [26]:
(
    df_bronze.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(bronze_table)
)

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 28, Finished, Available, Finished, False)

In [27]:
spark.sql("""
SELECT *
FROM bronze_netflix_titles
""")

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 29, Finished, Available, Finished, False)

DataFrame[show_id: string, type: string, title: string, director: string, cast: string, country: string, date_added: string, release_year: string, rating: string, duration: string, listed_in: string, description: string, ingestion_timestamp: timestamp, batch_id: string, source_file: string]

In [28]:
source_count = df.count()

bronze_count = spark.table(
    bronze_table
).count()

print(source_count)
print(bronze_count)

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 30, Finished, Available, Finished, False)

8806
8806


In [29]:
audit = [{
    "table_name": bronze_table,
    "record_count": bronze_count,
    "batch_id": batch_id
}]

audit_df = spark.createDataFrame(audit)

audit_df = audit_df.withColumn(
    "load_time",
    current_timestamp()
)

audit_df.write.mode("append").saveAsTable("bronze_audit")

StatementMeta(, 9bbf5cf1-1108-430b-ad8e-dd2b8795fd23, 31, Finished, Available, Finished, False)